# RobustWorld: NVIDIA Cosmos-Predict2 (Video2World 2B) on Colab

Predicts the second half (target) of each eval clip from its first half (context), then builds a
comparison grid against the ground truth. The sample is 5 clips committed in `data/eval/sample5`.

**Runtime:** **A100** (Colab Pro), ideally with High-RAM. NVIDIA lists ~32 GB of GPU memory for this model, so a T4 or L4 won't fit. At the native 93 frames and 960x960, expect several minutes per clip; `--num-frames 29` is much faster but outside the trained length.

**Secrets** (key icon in the left sidebar, then enable notebook access):
- `GITHUB_TOKEN`: a fine-grained GitHub token with read access to `FinPThomas/RobustWorld` (the repo is private).
- `HF_TOKEN`: a Hugging Face token. First accept the licence on [nvidia/Cosmos-Predict2-2B-Video2World](https://huggingface.co/nvidia/Cosmos-Predict2-2B-Video2World) and [nvidia/Cosmos-Guardrail1](https://huggingface.co/nvidia/Cosmos-Guardrail1) (both are gated).

Predictions are saved to Google Drive under `MyDrive/RobustWorld/predictions/`. The other model's
notebook writes to the same place, so the grid at the end includes every model that has been run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
REPO = "/content/RobustWorld"
if not os.path.exists(REPO):
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/FinPThomas/RobustWorld.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -e . -r models/cosmos_predict2/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PRED_ROOT = "/content/drive/MyDrive/RobustWorld/predictions"
GRID_ROOT = "/content/drive/MyDrive/RobustWorld/grids"
SAMPLE = "data/eval/sample5"
os.makedirs(PRED_ROOT, exist_ok=True)

## Run Cosmos-Predict2

Gets the last 5 context frames (its trained multi-frame conditioning) and rolls the scene
forward. The Cosmos guardrail stays on, as the NVIDIA Open Model License requires.

In [ ]:
!python models/cosmos_predict2/predict.py --sample {SAMPLE} --out {PRED_ROOT}/sample5/cosmos_predict2 --num-frames 93 --steps 35

## Comparison grid

Columns: ground truth, then every model with predictions in Drive. Rows: the input each source
was given (frames a model didn't see are dimmed), then its output. The context plays first,
then the predictions.

In [ ]:
import pathlib
order = ["cosmos_predict2", "wan21"]            # column order; add new models here
have = [m for m in order if (pathlib.Path(PRED_ROOT) / "sample5" / m).exists()]
out = f"{GRID_ROOT}/sample5/{'+'.join(have)}"
!robustworld-eval grid --sample {SAMPLE} --pred-root {PRED_ROOT} --models {" ".join(have)} --out {out}

In [ ]:
from IPython.display import Video, display
display(Video(f"{out}/all.mp4", embed=True, width=900))